In [1]:
!pip3 install pandas numpy scipy scikit-learn imbalanced-learn matplotlib seaborn
!pip3 install torch --index-url https://download.pytorch.org/whl/cu129
!pip3 install --extra-index-url=https://pypi.nvidia.com \
    "cuml-cu12==25.10.*"

Looking in indexes: https://download.pytorch.org/whl/cu129
Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com


In [2]:
import csv
import itertools
import joblib
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import pickle
import seaborn as sns

# scikit-learn
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,  
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import MinMaxScaler

# pytorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, TensorDataset, DataLoader

# rapids cuml
from cuml.ensemble import RandomForestClassifier as cuRFC
from cuml.ensemble import RandomForestRegressor as cuRFR

# set device for torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_state_datapoint_counts(df):
    """
    Return a DataFrame of states and their respective number of datetime records,
    sorted from most to least.
    """
    counts = (
        df.groupby('state')['datetime']
          .count()
          .reset_index(name='datapoint_count')
          .sort_values(by='datapoint_count', ascending=False)
    )
    return counts

# Example usage:
state_counts = get_state_datapoint_counts(orig_df)
print(state_counts)

def get_city_datapoint_counts(df):
    """
    Return a DataFrame of states and their respective number of datetime records,
    sorted from most to least.
    """
    counts = (
        df.groupby('city')['datetime']
          .count()
          .reset_index(name='datapoint_count')
          .sort_values(by='datapoint_count', ascending=False)
    )
    return counts

# Example usage:
state_counts = get_city_datapoint_counts(orig_df)
print(state_counts)

def get_place_datapoint_counts(df):
    """
    Return a DataFrame of states and their respective number of datetime records,
    sorted from most to least.
    """
    counts = (
        df.groupby('place')['datetime']
          .count()
          .reset_index(name='datapoint_count')
          .sort_values(by='datapoint_count', ascending=False)
    )
    return counts

# Example usage:
state_counts = get_place_datapoint_counts(orig_df)
print(state_counts)

def plot_state_month_heatmap(df, agg_col='temperature', agg_func='mean', scale=None):
    # Ensure month column exists
    if 'month' not in df.columns:
        df['month'] = df['datetime'].dt.month
    
    # Create pivot table: state vs month
    pivot_df = df.pivot_table(
        values=agg_col,
        index='state',
        columns='month',
        aggfunc=agg_func
    )
    
    scale_factor = 1
    scale_label = ""
    if scale == 'k':
        scale_factor = 1e3
        scale_label = " (k)"
    elif scale == 'M':
        scale_factor = 1e6
        scale_label = " (M)"
    
    pivot_df = pivot_df / scale_factor

    # Plot heatmap
    plt.figure(figsize=(12, 8))
    sns.heatmap(pivot_df, annot=True, cmap="coolwarm", fmt=".2f")
    plt.title(f"Heatmap of {agg_func} {agg_col}{scale_label} by State and Month", fontsize=14)
    plt.xlabel("Month")
    plt.ylabel("State")
    plt.show()

plot_state_month_heatmap(df_filtered, agg_col='temperature', agg_func='mean')
plot_state_month_heatmap(df_filtered, agg_col='precipitation_total', agg_func='mean')


In [3]:
orig_df = pd.read_csv('full_weather.csv', parse_dates=['datetime'])

def filter_top_place_per_state(df, state_col='state', place_col='place', datetime_col='datetime'):
    """
    Keep only the place with the highest datetime count for each state.

    Parameters:
    df (DataFrame): Input dataframe.
    state_col (str): Column name for state.
    place_col (str): Column name for place.
    datetime_col (str): Column name for datetime.

    Returns:
    DataFrame: Filtered dataframe.
    """
    # Count unique datetime values per state-place
    counts = (
        df.groupby([state_col, place_col])[datetime_col]
        .nunique()
        .reset_index(name='datetime_count')
    )

    # Get the place with the max datetime_count per state
    top_places = counts.loc[counts.groupby(state_col)['datetime_count'].idxmax()]

    # Merge back to original df
    return df.merge(top_places[[state_col, place_col]], on=[state_col, place_col])

df_filtered = filter_top_place_per_state(orig_df, state_col='state', place_col='place', datetime_col='datetime')

## Pre-processing datasets

In [4]:
pre_df = df_filtered

# clear unnecessary columns
pre_df = pre_df.drop(columns=['pollutant_value'])

### Seasonal Interpolation of Missing Values
# Fill missing values per feature using monthly means.
pre_df['month'] = pre_df['datetime'].dt.month
features = [
    'pressure','dew_point','humidity','wind_speed',
    'gust','wind_chill','uv_index','feels_like_temperature',
    'visibility','solar_radiation']

month_means = pre_df.groupby('month')[features].mean()

# Fill NaNs by mapping month to mean values
for feat in features:
    na_mask = pre_df[feat].isna()
    pre_df.loc[na_mask, feat] = pre_df.loc[na_mask, 'month'].map(month_means[feat])

### Outlier Removal (Z‑score ±3)
# z_scores = df[features].apply(zscore)
# mask = (np.abs(z_scores) <= 3).all(axis=1)
# print(f"Removing {len(df) - mask.sum()} outliers.")
# df = df[mask].reset_index(drop=True)


### Feature Engineering
# - 1-hour, 1-day rolling average of precipitation and temperature
# - Monsoon month flag (Nov–Mar)
# - Coastal state flag
pre_df = pre_df.sort_values(['place','datetime']).reset_index(drop=True)

def fill_missing_precipitation(group):
    for col in ["precipitation_rate", "precipitation_total"]:
        # Mark where it rained
        rain_mask = group[col].fillna(0) > 0

        # Last rain datetime (ffill inside each place group)
        group["last_rain_time"] = group["datetime"].where(rain_mask).ffill()

        # Last rain value (ffill inside group)
        group["last_rain_value"] = group[col].where(rain_mask).ffill()

        # Missing rows
        missing_mask = group[col].isna()
        within_30min = (group["datetime"] - group["last_rain_time"]) <= pd.Timedelta("30min")

        # Fill with last rain value or 0
        group.loc[missing_mask & within_30min, col] = group.loc[missing_mask & within_30min, "last_rain_value"]
        group.loc[missing_mask & ~within_30min, col] = 0

        # Drop helper cols
        group = group.drop(columns=["last_rain_time", "last_rain_value"])

    return group

pre_df = pre_df.groupby("place", group_keys=False)[pre_df.columns].apply(fill_missing_precipitation)

# linear interpolation on missing temperature values
pre_df['temperature'] = pre_df['temperature'].interpolate(method='linear', limit_direction='both')

# 7‑day rolling avg per place
pre_df['precipitation_total_1h_avg'] = (
    pre_df.groupby("place")
      .rolling("1h", on="datetime")["precipitation_total"]
      .mean()
      .reset_index(drop=True)
)

pre_df['precipitation_total_1d_avg'] = (
    pre_df.groupby("place")
      .rolling("1d", on="datetime")["precipitation_total"]
      .mean()
      .reset_index(drop=True)
)

pre_df['precipitation_rate_1h_avg'] = (
    pre_df.groupby("place")
      .rolling("1h", on="datetime")["precipitation_rate"]
      .mean()
      .reset_index(drop=True)
)

pre_df['precipitation_rate_1d_avg'] = (
    pre_df.groupby("place")
      .rolling("1d", on="datetime")["precipitation_rate"]
      .mean()
      .reset_index(drop=True)
)

# 1‑hour rolling avg of temperature
pre_df['temperature_1h_avg'] = (
    pre_df.groupby("place")
      .rolling("1h", on="datetime")["temperature"]
      .mean()
      .reset_index(drop=True)
)

pre_df['temperature_1d_avg'] = (
    pre_df.groupby("place")
      .rolling("1d", on="datetime")["temperature"]
      .mean()
      .reset_index(drop=True)
)

# additional features
additional_features = [
    'temperature', 'precipitation_rate', 'precipitation_total',
    'precipitation_total_1h_avg', 'precipitation_total_1d_avg',
    'precipitation_rate_1h_avg', 'precipitation_rate_1d_avg',
    'temperature_1h_avg', 'temperature_1d_avg'
]

# Monsoon flag
pre_df['is_monsoon'] = pre_df['month'].apply(lambda m: 1 if m in [11,12,1,2,3] else 0)

# Coastal flag
coastal_states = ['Penang','Kedah','Selangor','Johor','Kelantan','Terengganu','Sarawak','Sabah']
pre_df['is_coastal'] = pre_df['state'].isin(coastal_states).astype(int)


### Rainfall flag
# if there is precipitation evaluated, it means there is rain
pre_df['rain_label'] = (pre_df['precipitation_rate'] > 0).astype(int)

### Scaling (Min‑Max)
scaler = MinMaxScaler()
cols_to_scale = features + additional_features

# Apply Min-Max scaling
scaled_values = scaler.fit_transform(pre_df[cols_to_scale])
pre_df[cols_to_scale] = scaled_values


## Temporal Split
Train: 1996–2013; Validation: 2014–2021; Test: 2022–present

In [5]:
pre_df['year'] = pre_df['datetime'].dt.year

del orig_df
del df_filtered

train_df = pre_df[pre_df['year'].between(1996,2014)].copy()
val_df   = pre_df[pre_df['year'].between(2014,2021)].copy()
test_df  = pre_df[pre_df['year'] >= 2022].copy()

del pre_df

print(f"Train: {len(train_df)} rows | Val: {len(val_df)} | Test: {len(test_df)}")

print("train_df rain_label distribution:")
print(train_df['rain_label'].value_counts(dropna=False))
print("val_df rain_label distribution:")
print(val_df['rain_label'].value_counts(dropna=False))
print("test_df rain_label distribution:")
print(test_df['rain_label'].value_counts(dropna=False))

Train: 851893 rows | Val: 1952666 | Test: 2928737
train_df rain_label distribution:
rain_label
0    843311
1      8582
Name: count, dtype: int64
val_df rain_label distribution:
rain_label
0    1868617
1      84049
Name: count, dtype: int64
test_df rain_label distribution:
rain_label
0    2778115
1     150622
Name: count, dtype: int64


## Helper function

In [6]:
def generate_csv(model, results):
    df_results = pd.DataFrame(results)
    df_results.to_csv(f"{model}_results.csv", index=False)

def load_or_create_results_df(filename, param_keys, metric_keys):
    filepath = f"{filename}.csv"
    if os.path.exists(filepath):
        return pd.read_csv(filepath)
    else:
        columns = list(param_keys) + metric_keys
        return pd.DataFrame(columns=columns)

def check_if_run_exists(df, param_dict):
    if df.empty:
        return False

    mask = pd.Series(True, index=df.index)
    for key, value in param_dict.items():
        if value is None:
            mask &= df[key].isna()
        else:
            mask &= (df[key] == value)
            
    return mask.any()

def append_result_to_csv(filename, result_dict):
    filepath = f"{filename}.csv"
    file_exists = os.path.isfile(filepath)
    with open(filepath, mode='a', newline='') as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=result_dict.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(result_dict)

def get_best_hyperparameters_from_csv(filename, metric_column, mode):
    filepath = f"{filename}.csv"
    if not os.path.exists(filepath):
        print(f"Results file not found.")
        return None

    try:
        df = pd.read_csv(filepath)
        if df.empty:
            print(f"Results file is empty.")
            return None
        df[metric_column] = pd.to_numeric(df[metric_column])
        if mode == 'min':
            best_run_index = df[metric_column].idxmin()
        elif mode == 'max':
            best_run_index = df[metric_column].idxmax()
        else:
            raise ValueError("Mode must be either 'min' or 'max'.")
        best_run = df.loc[best_run_index]
        return best_run.to_dict()

    except (KeyError, ValueError) as e:
        print(f"Error processing file : {e}")
        return None

def get_best_lstm_hyperparameters(filename, acc_col = 'val_accuracy',rmse_col = 'val_rmse', weight = 0.5):
    filepath = f"{filename}.csv"

    if not os.path.exists(filepath):
        print(f"Results file '{filepath}' not found.")
        return None

    df = pd.read_csv(filepath)
    if df.empty:
        print(f"Results file '{filepath}' is empty.")
        return None

    acc_min, acc_max = df[acc_col].min(), df[acc_col].max()
    if acc_max == acc_min:
        df['norm_acc'] = 0.5
    else:
        df['norm_acc'] = (df[acc_col] - acc_min) / (acc_max - acc_min)

    rmse_min, rmse_max = df[rmse_col].min(), df[rmse_col].max()
    if rmse_max == rmse_min:
        df['norm_rmse'] = 0.5
    else:
        df['norm_rmse'] = (df[rmse_col] - rmse_min) / (rmse_max - rmse_min)

    df['performance_score'] = (weight * df['norm_acc']) + ((1 - weight) * (1 - df['norm_rmse']))

    best_run_index = df['performance_score'].idxmax()
    best_run = df.loc[best_run_index]
    return best_run.to_dict()

def save_with_pickle(obj, filename):
    with open(filename, "wb") as f:
        pickle.dump(obj, f)

def load_with_pickle(filename):
    with open(filename, "rb") as f:
        return pickle.load(f)

def regression_plot(y_labels, y_preds):
    y_true = np.array(y_labels).flatten()
    y_pred = np.array(y_preds).flatten()
    residuals = y_true - y_pred

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    sns.scatterplot(x=y_true, y=y_pred, alpha=0.6, ax=ax1)
    ax1.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--')
    ax1.set_xlabel("Actual")
    ax1.set_ylabel("Predicted")
    ax1.set_title("Scatter Plot of Actual vs Predicted")

    sns.scatterplot(x=y_pred, y=residuals, alpha=0.6, ax=ax2)
    ax2.axhline(0, color='red', linestyle='--')
    ax2.set_xlabel("Predicted")
    ax2.set_ylabel("Residuals")
    ax2.set_title("Residual Plot of Residuals vs Predicted")

    plt.tight_layout()
    plt.show()

def regression_results(y_labels, y_preds):
    mse = mean_squared_error(y_labels, y_preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_labels, y_preds)

    print(f"Test MSE:  {mse:.4f}")
    print(f"Test RMSE: {rmse:.4f}")
    print(f"Test R2:  {r2:.4f}")

def classification_results(y_labels, y_preds):
    accuracy = accuracy_score(y_labels, y_preds)
    precision = precision_score(y_labels, y_preds)
    recall = recall_score(y_labels, y_preds)
    f1 = f1_score(y_labels, y_preds)

    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")

    print("Classification Report:")
    print(classification_report(y_labels, y_preds))

def classification_plot(y_labels, y_preds):

    # Create confusion matrix and classification report
    cm = confusion_matrix(y_labels, y_preds)
    report = classification_report(y_labels, y_preds, output_dict=True)
    df = pd.DataFrame(report).transpose()
    df = df.drop(['accuracy'], errors='ignore')

    # Create figure with two subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(ax=ax1, cmap='Blues', values_format='d')
    ax1.set_title("Confusion Matrix")
    sns.heatmap(df.iloc[:-1, :3], annot=True, cmap='YlGnBu', fmt=".2f", ax=ax2)
    ax2.set_title("Classification Report Heatmap")

    plt.tight_layout()
    plt.show()

## Temperature Linear Regression

In [7]:
class TemperatureLinearRegressionModel(torch.nn.Module):
    '''A simple linear regression model with one output.'''
    def __init__(self, input_dim, output_dim):
        super(TemperatureLinearRegressionModel, self).__init__()
        self.linear = torch.nn.Linear(input_dim, output_dim)

    def forward(self, x):
        y_pred = self.linear(x)
        return y_pred


def temperature_linear_train_val_dataset(features, labels, batch_size, num_workers=0):
    X_train = torch.tensor(train_df[features].values, dtype=torch.float32)
    y_train = torch.tensor(train_df[labels].values, dtype=torch.float32).unsqueeze(1)

    X_val = torch.tensor(val_df[features].values, dtype=torch.float32)
    y_val = torch.tensor(val_df[labels].values, dtype=torch.float32).unsqueeze(1)

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=batch_size, num_workers=num_workers)

    return train_loader, val_loader

def temperature_linear_test_dataset(features, labels, batch_size=128, num_workers=0):
    X_test = torch.tensor(test_df[features].values, dtype=torch.float32)
    y_test = torch.tensor(test_df[labels].values, dtype=torch.float32).unsqueeze(1)

    test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, num_workers=num_workers)

    return test_loader

def train_temperature_linear_model(model, train_loader, val_loader, epochs, lr):

    history = {
        'train_loss': [],
        'train_rmse': [],
        'val_loss': [],
        'val_rmse': []
    }

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        train_loss, batch_rmse, train_total = 0, 0, 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            output = model(xb)
            loss = torch.sqrt(criterion(output, yb))
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            batch_rmse = torch.sqrt(torch.mean((output - yb) ** 2))
            train_total += batch_rmse.item()

        train_loss /= len(train_loader)
        train_rmse = train_total / len(train_loader)
        history['train_loss'].append(train_loss)
        history['train_rmse'].append(train_rmse)

        model.eval()
        val_loss, val_rmse, val_total = 0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)

                preds = model(xb)
                loss = torch.sqrt(criterion(preds, yb))

                val_loss += loss.item()

                batch_rmse = torch.sqrt(torch.mean((preds - yb) ** 2))
                val_total += batch_rmse.item()

        val_loss /= len(val_loader)
        val_rmse = val_total / len(val_loader)
        history['val_loss'].append(val_loss)
        history['val_rmse'].append(val_rmse)

        print(f"Epoch {epoch}/{epochs}, Train Loss: {train_loss:.4f}, Train RMSE: {train_rmse:.4f},  Val Loss: {val_loss:.4f}, Val RMSE: {val_rmse:.4f}")

    return model, history
        
def evaluate_temperature_linear_model(model, test_loader):
    y_labels = []
    y_preds = []

    model.eval()
    test_loss, test_rmse, test_total = 0, 0, 0

    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)

            outputs = model(xb)

            y_labels.extend(yb.cpu().numpy())
            y_preds.extend(outputs.cpu().numpy())

    regression_results(y_labels, y_preds)
    regression_plot(y_labels, y_preds)

    return y_labels, y_preds

def plot_temperature_linear_model_history(history):
    epochs_range = range(1, len(history['train_loss']) + 1)

    plt.figure(figsize=(12, 5))

    # Plot Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, history['train_loss'], label='Training Loss')
    plt.plot(epochs_range, history['val_loss'], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Loss over Epochs')
    plt.legend()
    plt.grid(True)

    # Plot Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, history['train_rmse'], label='Training Accuracy')
    plt.plot(epochs_range, history['val_rmse'], label='Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.title('Accuracy over Epochs')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

def temperature_linear_model_save(model, filename_path):
    torch.save(model.state_dict(), filename_path)

def temperature_linear_model_load(hyperparameters, filename_path):
    saved_model = TemperatureLinearRegressionModel(
        input_dim=hyperparameters['input_size'], 
        output_dim=hyperparameters['output_size']
    )
    saved_model.load_state_dict(torch.load(filename_path, weights_only=True), strict=False)
    return saved_model

In [8]:
def temperature_linear_model_gridsearch(features, labels, param_grid=None):
    '''Gridsearch over hyperparameters'''
    results = []
    best_model, best_params, best_result = None, None, None
    best_score = float("inf")  # lower RMSE is better

    for params in itertools.product(*param_grid.values()):
        param_dict = dict(zip(param_grid.keys(), params))

        train_loader, val_loader = temperature_linear_train_val_dataset(
            features, labels, 
            batch_size=param_dict['batch_size'], 
            num_workers=0
        )

        model = TemperatureLinearRegressionModel(
            input_dim=len(features), 
            output_dim=1
        ).to(device)

        model, model_result = train_temperature_linear_model(
            model, train_loader, val_loader, 
            epochs=param_dict['epochs'], 
            lr=param_dict['lr']
        )

        last_val_rmse = model_result['val_rmse'][-1]

        param_dict.update({
            'train_rmse': model_result['train_rmse'][-1],
            'val_rmse': model_result['val_rmse'][-1]
        })

        results.append(param_dict)

        # Keep only the best model
        if last_val_rmse < best_score:
            best_score = last_val_rmse
            best_params = param_dict
            best_result = model_result
            best_model = model  # keep this model


    # best_params, best_result = min(results, key=lambda x: min(x[1]['val_rmse']))
    print("\nBest Params:", best_params)
    print("Best Model Result:", best_result)

    temperature_linear_model_save(best_model, 'temperature_linear_model.pth')

    # Plot the model loss
    plot_temperature_linear_model_history(best_result)

    # Evaluate on test set
    test_loader = temperature_linear_test_dataset(
        features, labels, 
        batch_size=max(param_grid['batch_size'], key=lambda x: x), 
        num_workers=0
    )

    generate_csv('temperature_linear_model', results)
    
    y_labels, y_preds = evaluate_temperature_linear_model(best_model, test_loader)

    return best_model, y_labels, y_preds

## Rainfall binary classification

In [9]:
class RainBinaryClassifierModel(nn.Module):
    def __init__(self, input_size, hidden_size=64):
        super(RainBinaryClassifierModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)  # Single output neuron
        )
        
    def forward(self, x):
        return self.net(x)  # raw logits

def rainfall_binary_train_val_dataset(features, labels, batch_size=128, num_workers=0):
    X_train = torch.tensor(train_df[features].values, dtype=torch.float32)
    y_train = torch.tensor(train_df[labels].values, dtype=torch.float32).unsqueeze(1)

    X_val = torch.tensor(val_df[features].values, dtype=torch.float32)
    y_val = torch.tensor(val_df[labels].values, dtype=torch.float32).unsqueeze(1)

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=batch_size, num_workers=num_workers)

    return train_loader, val_loader

def rainfall_binary_test_dataset(features, labels, batch_size=128, num_workers=0):
    X_test = torch.tensor(test_df[features].values, dtype=torch.float32)
    y_test = torch.tensor(test_df[labels].values, dtype=torch.float32).unsqueeze(1)

    test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, num_workers=num_workers)

    return test_loader

def train_rainfall_binary_model(model, train_loader, val_loader, epochs, lr, class_weight=False):

    history = {
        'train_loss': [],
        'train_accuracy': [],
        'val_loss': [],
        'val_accuracy': []
    }

    if class_weight:
        y_train = train_df['rain_label']
        num_pos = (y_train == 1).sum()
        num_neg = (y_train == 0).sum()

        pos_weight = num_neg / num_pos
        criterion = nn.BCEWithLogitsLoss(pos_weight = torch.tensor(pos_weight, dtype=torch.float32))
    else:
        criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            outputs = model(xb)
            loss = criterion(outputs, yb)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * xb.size(0)

            # Predictions for accuracy
            predicted_labels = (torch.sigmoid(outputs) >= 0.5).float()
            train_correct += (predicted_labels == yb).sum().item()
            train_total += yb.size(0)

        train_loss /= train_total
        train_acc = train_correct / train_total
        history['train_loss'].append(train_loss)
        history['train_accuracy'].append(train_acc)

        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)

                outputs = model(xb)
                loss = criterion(outputs, yb)

                val_loss += loss.item() * xb.size(0)

                # Calculate accuracy
                predicted_labels = (torch.sigmoid(outputs) >= 0.5).float()
                val_correct += (predicted_labels == yb).sum().item()
                val_total += yb.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_acc)

        print(f"Epoch {epoch}/{epochs}, Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.4f},  Val Loss: {val_loss:.4f}, Val Accuracy: {val_acc:.4f}")
    
    return model, history

def evaluate_rainfall_binary_model(model, test_loader):
    model.eval()  # evaluation mode
    y_preds = []
    y_labels = []
    test_correct,test_total = 0, 0

    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)

            outputs = model(xb)

            predicted_labels = (torch.sigmoid(outputs) >= 0.5).float()
            test_correct += (predicted_labels == yb).sum().item()
            test_total += yb.size(0)

            y_preds.extend(predicted_labels.cpu().numpy())
            y_labels.extend(yb.cpu().numpy())

    test_accuracy = 100 * test_correct / test_total
    print(f"Test Accuracy: {test_accuracy:.2f}%")

    classification_results(y_labels, y_preds)
    classification_plot(y_labels, y_preds)

    return y_labels, y_preds

def plot_rainfall_binary_model_history(history):
    epochs_range = range(1, len(history['train_loss']) + 1)

    plt.figure(figsize=(12, 5))

    # Plot Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, history['train_loss'], label='Training Loss')
    plt.plot(epochs_range, history['val_loss'], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Loss over Epochs')
    plt.legend()
    plt.grid(True)

    # Plot Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, history['train_accuracy'], label='Training Accuracy')
    plt.plot(epochs_range, history['val_accuracy'], label='Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.title('Accuracy over Epochs')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

def rainfall_classifier_model_save(model, filename_path):
    torch.save(model.state_dict(), filename_path)

def rainfall_classifier_model_load(hyperparameters, filename_path):
    saved_model = RainBinaryClassifierModel(
        input_size=hyperparameters['input_size'] 
    )
    saved_model.load_state_dict(torch.load(filename_path, weights_only=True), strict=False)
    return saved_model

In [10]:
def rainfall_binary_model_gridsearch(features, labels, param_grid=None):
    '''Gridsearch over hyperparameters'''
    results = []
    best_model, best_params, best_result = None, None, None
    best_accuracy = 0

    for params in itertools.product(*param_grid.values()):
        param_dict = dict(zip(param_grid.keys(), params))
        
        train_loader, val_loader = rainfall_binary_train_val_dataset(
            features, labels, 
            batch_size=param_dict['batch_size'], 
            num_workers=0
        )

        model = RainBinaryClassifierModel(
            input_size=len(features), 
        ).to(device)

        model, model_result = train_rainfall_binary_model(
            model, train_loader, val_loader, 
            epochs=param_dict['epochs'], 
            lr=param_dict['lr'],
            class_weight=param_dict['class_weight']
        )

        last_val_acc = model_result['val_accuracy'][-1]

        param_dict.update({
            'train_accuracy': model_result['train_accuracy'][-1],
            'val_accuracy': model_result['val_accuracy'][-1]
        })

        results.append(param_dict)

        # Keep only the best model
        if last_val_acc > best_accuracy:
            best_accuracy = last_val_acc
            best_params = param_dict
            best_result = model_result
            best_model = model

    # best_params, best_result = min(results, key=lambda x: min(x[1]['val_rmse']))
    print("\nBest Params:", best_params)
    print("Best Model Result:", best_result)

    generate_csv('rainfall_classifier_model', results)

    # Plot the model loss
    plot_rainfall_binary_model_history(model_result)

    # Evaluate on test set
    test_loader = rainfall_binary_test_dataset(
        features, labels, 
        batch_size=max(param_grid['batch_size'], key=lambda x: x), 
        num_workers=0
    )

    y_labels, y_preds = evaluate_rainfall_binary_model(best_model, test_loader)

    rainfall_classifier_model_save(best_model, 'rainfall_classifier_model.pth')

    return best_model, y_labels, y_preds

## Long Short-Term Memory

In [11]:
class WeatherLSTMDataset(Dataset):
    def __init__(self, X, y_temp, y_rain, sequence_length=7):
        self.X = X
        self.y_temp = y_temp
        self.y_rain = y_rain
        self.seq_len = sequence_length

    def __len__(self):
        return len(self.X) - self.seq_len

    def __getitem__(self, idx):
        X_seq = self.X[idx:idx+self.seq_len]
        y_temp = self.y_temp[idx+self.seq_len]
        y_rain = self.y_rain[idx+self.seq_len]
        return X_seq, (y_temp, y_rain)


class WeatherLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super(WeatherLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0  # Dropout only works with >1 layer
        )
        self.dropout = nn.Dropout(dropout)
        self.fc_temp = nn.Linear(hidden_size, 1)
        self.fc_rain  = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]   # take last time step
        out = self.dropout(out)
        temp_out = self.fc_temp(out)   # regression (raw float)
        rain_out = self.fc_rain(out)   # classification (0–1 prob)

        return temp_out, rain_out  # return both outputs

def weather_lstm_train_val_dataset(features, label_1, label_2):
    X_train = torch.tensor(train_df[features].values, dtype=torch.float32)
    y_temp_train = torch.tensor(train_df[label_1].values, dtype=torch.float32).unsqueeze(1)
    y_rain_train = torch.tensor(train_df[label_2].values, dtype=torch.float32).unsqueeze(1)

    X_val = torch.tensor(val_df[features].values, dtype=torch.float32)
    y_temp_val = torch.tensor(val_df[label_1].values, dtype=torch.float32).unsqueeze(1)
    y_rain_val = torch.tensor(val_df[label_2].values, dtype=torch.float32).unsqueeze(1)

    return X_train, y_temp_train, y_rain_train, X_val, y_temp_val, y_rain_val

def weather_lstm_test_dataset(features, label_1, label_2, batch_size=128, num_workers=0):
    X_test = torch.tensor(test_df[features].values, dtype=torch.float32)
    y_temp_test = torch.tensor(test_df[label_1].values, dtype=torch.float32).unsqueeze(1)
    y_rain_test = torch.tensor(test_df[label_2].values, dtype=torch.float32).unsqueeze(1)

    test_loader = DataLoader(WeatherLSTMDataset(X_test, y_temp_test, y_rain_test), batch_size=batch_size, num_workers=num_workers)

    return test_loader

# train a single model for both temperature and rainfall prediction
def multitask_loss(temp_pred, temp_true, rain_pred, rain_true, mse_loss, bce_loss, alpha=0.5):
    loss_temp = torch.sqrt(mse_loss(temp_pred, temp_true))
    loss_rain = bce_loss(rain_pred, rain_true)
    return alpha * loss_temp + (1 - alpha) * loss_rain

def train_weather_lstm_model(model, train_loader, val_loader, epochs, lr, class_weight=False):

    history = {
        'train_loss': [],
        'train_accuracy': [],
        'train_rmse': [],
        'val_loss': [],
        'val_accuracy': [],
        'val_rmse': []
    }

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    mse_loss = nn.MSELoss()

    # Handle class imbalance for rainfall classification
    if class_weight:
        y_train = train_df['rain_label']
        num_pos = (y_train == 1).sum()
        num_neg = (y_train == 0).sum()

        pos_weight = num_neg / num_pos
        bce_loss = nn.BCEWithLogitsLoss(pos_weight = torch.tensor(pos_weight, dtype=torch.float32))
    else:
        bce_loss = nn.BCEWithLogitsLoss()

    # training loop
    for epoch in range(epochs):
        model.train()
        train_loss, train_batch_rmse, train_rmse_total, train_correct, train_total, train_rmse = 0, 0, 0, 0, 0, 0

        for xb, (y_temp, y_rain) in train_loader:
            xb, y_temp, y_rain = xb.to(device), y_temp.to(device), y_rain.to(device)

            optimizer.zero_grad()
            temp_output, rain_output = model(xb)
            loss = multitask_loss(temp_output, y_temp, rain_output, y_rain, mse_loss, bce_loss)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            # Calculate RMSE for temperature
            train_batch_rmse = torch.sqrt(torch.mean((temp_output - y_temp) ** 2))
            train_rmse_total += train_batch_rmse.item()

            # Calculate accuracy for rainfall
            predicted_labels = (torch.sigmoid(rain_output) >= 0.5).float()
            train_correct += (predicted_labels == y_rain).sum().item()
            train_total += y_rain.size(0)

        train_loss /= train_total
        train_acc = train_correct / train_total
        train_rmse = train_rmse_total / train_total

        history['train_loss'].append(train_loss)
        history['train_accuracy'].append(train_acc)
        history['train_rmse'].append(train_rmse)

        # Validation
        model.eval()

        val_loss, val_batch_rmse, val_rmse_total, val_correct, val_total, val_rmse = 0, 0, 0, 0, 0, 0
        with torch.no_grad():
            for xb, (y_temp, y_rain) in val_loader:
                xb, y_temp, y_rain = xb.to(device), y_temp.to(device), y_rain.to(device)

                temp_output, rain_output = model(xb)
                loss = multitask_loss(temp_output, y_temp, rain_output, y_rain, mse_loss, bce_loss)

                val_loss += loss.item()

                # Calculate RMSE for temperature
                val_batch_rmse = torch.sqrt(torch.mean((temp_output - y_temp) ** 2))
                val_rmse_total += val_batch_rmse.item()

                # Calculate accuracy for rainfall
                predicted_labels = (torch.sigmoid(rain_output) >= 0.5).float()
                val_correct += (predicted_labels == y_rain).sum().item()
                val_total += y_rain.size(0)

        val_loss /= val_total
        val_acc = val_correct / val_total
        val_rmse = val_rmse_total / val_total
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_acc)
        history['val_rmse'].append(val_rmse)
        
        print(f"\tEpoch {epoch + 1}/{epochs}, Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.4f}, Train RMSE: {train_rmse:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_acc:.4f}, Val RMSE: {val_rmse:.4f}")

    return model, history, optimizer

def evaluate_weather_lstm_model(model, test_loader):
    model.eval()  # evaluation mode
    y_temp_preds = []
    y_rain_preds = []
    y_temp_labels = []
    y_rain_labels = []
    
    test_correct,test_total = 0, 0
    
    with torch.no_grad():
        for xb, (y_temp, y_rain) in test_loader:
            xb, y_temp, y_rain = xb.to(device), y_temp.to(device), y_rain.to(device)

            temp_output, rain_output = model(xb)

            predicted_labels = (torch.sigmoid(rain_output) >= 0.5).float()
            test_correct += (predicted_labels == y_rain).sum().item()
            test_total += y_rain.size(0)
            
            y_rain_preds.extend(predicted_labels.cpu().numpy())
            y_rain_labels.extend(y_rain.cpu().numpy())

            y_temp_labels.extend(y_temp.cpu().numpy())
            y_temp_preds.extend(temp_output.cpu().numpy())


    regression_results(y_temp_labels, y_temp_preds)
    regression_plot(y_temp_labels, y_temp_preds)

    test_accuracy = 100 * test_correct / test_total
    print(f"Test Accuracy: {test_accuracy:.2f}%")

    classification_results(y_rain_labels, y_rain_preds)
    classification_plot(y_rain_labels, y_rain_preds)

    return y_temp_labels, y_temp_preds, y_rain_labels, y_rain_preds

def plot_rainfall_lstm_model_history(history):
    epochs_range = range(1, len(history['train_loss']) + 1)

    plt.figure(figsize=(12, 5))

    # Plot Loss
    plt.subplot(1, 3, 1)
    plt.plot(epochs_range, history['train_loss'], label='Training Model Loss')
    plt.plot(epochs_range, history['val_loss'], label='Validation Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Loss over Epochs')
    plt.legend()
    plt.grid(True)

    # Plot Accuracy
    plt.subplot(1, 3, 2)
    plt.plot(epochs_range, history['train_accuracy'], label='Training Rainfall Accuracy')
    plt.plot(epochs_range, history['val_accuracy'], label='Validation Rainfall Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.title('Accuracy over Epochs')
    plt.legend()
    plt.grid(True)

    # Plot RMSE
    plt.subplot(1, 3, 3)
    plt.plot(epochs_range, history['train_rmse'], label='Training Temperature RMSE')
    plt.plot(epochs_range, history['val_rmse'], label='Validation Temperature RMSE')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Loss over Epochs')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

def lstm_weather_model_save(checkpoint, filename_path):
    torch.save(checkpoint, filename_path)

def lstm_weather_model_save_2(model, filename_path):
    torch.save(model, filename_path)

def lstm_weather_model_load(filename_path):
    checkpoint = torch.load(filename_path)
    hyperparams = checkpoint["hyperparams"]

    model = WeatherLSTM(
        input_size=hyperparams['input_size'], 
        hidden_size=hyperparams['hidden_size'], 
        num_layers=hyperparams['num_layers'], 
        dropout=hyperparams['dropout']
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=hyperparams["lr"])
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    return model, optimizer, checkpoint["epoch"], checkpoint["history"], hyperparams

In [12]:
def weather_lstm_model_gridsearch(features, label_1, label_2, param_grid=None):
    
    lstm_iterator_counter = 0
    num_runs = np.prod([len(v) for v in param_grid.values()])
    lstm_df = load_or_create_results_df(
        'lstm_model_results',
        param_grid.keys(),
        ['train_accuracy', 'val_accuracy', 'train_rmse', 'val_rmse']
    )

    X_train, y_temp_train, y_rain_train, X_val, y_temp_val, y_rain_val = weather_lstm_train_val_dataset(features, label_1, label_2)

    for params in itertools.product(*param_grid.values()):
        param_dict = dict(zip(param_grid.keys(), params))

        lstm_iterator_counter += 1

        if check_if_run_exists(lstm_df, param_dict):
            print(f"Run {lstm_iterator_counter}/{num_runs}: skipping existing lstm params.")
            continue

        train_dataset = WeatherLSTMDataset(X_train, y_temp_train, y_rain_train, param_dict['sequence_length'])
        val_dataset = WeatherLSTMDataset(X_val, y_temp_val, y_rain_val, param_dict['sequence_length'])

        train_loader = DataLoader(train_dataset, batch_size=param_dict['batch_size'], shuffle=False) # no shuffle for LSTM
        val_loader = DataLoader(val_dataset, batch_size=param_dict['batch_size'], shuffle=False)

        model = WeatherLSTM(
            input_size=len(features), 
            hidden_size=param_dict['hidden_size'], 
            num_layers=param_dict['num_layers'], 
            dropout=param_dict['dropout']
        ).to(device)

        model, model_result = train_weather_lstm_model(
            model, train_loader, val_loader, 
            epochs=param_dict['epochs'], 
            lr=param_dict['lr'],
            class_weight=param_dict['class_weight']
        )
        
        param_dict.update({
            'train_accuracy': model_result['train_accuracy'][-1],
            'val_accuracy': model_result['val_accuracy'][-1],
            'train_rmse': model_result['train_rmse'][-1],
            'val_rmse': model_result['val_rmse'][-1]
        })

        append_result_to_csv('lstm_model_result', param_dict)

        print(f'Run {lstm_iterator_counter}/{num_runs} completed for lstm search')

def weather_lstm_model_reevaluate(features, label_1, label_2):

    X_train, y_temp_train, y_rain_train, X_val, y_temp_val, y_rain_val = weather_lstm_train_val_dataset(features, label_1, label_2)
    test_loader = weather_lstm_test_dataset(features, label_1, label_2,batch_size=1024,num_workers=0)

    best_lstm_run = get_best_lstm_hyperparameters(
        'lstm_model_results',
        weight=0.5
    )

    train_dataset = WeatherLSTMDataset(X_train, y_temp_train, y_rain_train, best_lstm_run['sequence_length'])
    val_dataset = WeatherLSTMDataset(X_val, y_temp_val, y_rain_val, best_lstm_run['sequence_length'])

    train_loader = DataLoader(train_dataset, batch_size=best_lstm_run['batch_size'], shuffle=False) # no shuffle for LSTM
    val_loader = DataLoader(val_dataset, batch_size=best_lstm_run['batch_size'], shuffle=False)

    lstm_model = WeatherLSTM(
        input_size=len(features), 
        hidden_size=best_lstm_run['hidden_size'], 
        num_layers=best_lstm_run['num_layers'], 
        dropout=best_lstm_run['dropout']
    ).to(device)

    best_lstm_model, _ = train_weather_lstm_model(
        lstm_model, train_loader, val_loader, 
        epochs=best_lstm_run['epochs'], 
        lr=best_lstm_run['lr'],
        class_weight=best_lstm_run['class_weight']
    )

    y_temp_labels, y_temp_preds, y_rain_labels, y_rain_preds = evaluate_weather_lstm_model(best_lstm_model, test_loader)
    
    lstm_model_dict = {
        'lstm_model': best_lstm_model,
        'lstm_y_temp_labels': y_temp_labels,
        'lstm_y_temp_preds': y_temp_preds,
        'lstm_y_rain_labels': y_rain_labels,
        'lstm_y_rain_preds': y_rain_preds,
        'lstm_hyperparameters': best_lstm_run
    }

    save_with_pickle(lstm_model_dict, 'lstm_model_results.pkl')

    return lstm_model_dict

## Random Forest

In [13]:
def weather_rf_train_val_dataset(features, label_1, label_2):
    X_train = train_df[features].values
    y_temp_train = train_df[label_1].values
    y_rain_train = train_df[label_2].values

    X_val = val_df[features].values
    y_temp_val = val_df[label_1].values
    y_rain_val = val_df[label_2].values

    return X_train, y_temp_train, y_rain_train, X_val, y_temp_val, y_rain_val

def weather_rf_test_dataset(features, label_1, label_2):
    X_test = test_df[features].values
    y_temp_test = test_df[label_1].values
    y_rain_test = test_df[label_2].values

    return X_test, y_temp_test, y_rain_test

def train_weather_rf_regressor_model(
        X_train, y_train_temp, 
        X_val, y_temp_val, 
        n_estimators, regressor_criterion, 
        max_depth, min_samples_split, min_samples_leaf, 
        max_features, max_samples, bootstrap, rf_flag='scikit'):

    if rf_flag == 'cuml':
        rf_temp = cuRFR(
            n_estimators = n_estimators,
            split_criterion = regressor_criterion,
            # max_depth = max_depth,
            min_samples_split = min_samples_split,
            min_samples_leaf = min_samples_leaf,
            max_features = max_features,
            max_samples = max_samples,
            bootstrap = bootstrap,
            random_state = 42
        )
    else:
        rf_temp = RandomForestRegressor(
            n_estimators = n_estimators,
            criterion = regressor_criterion,
            max_depth = max_depth,
            min_samples_split = min_samples_split,
            min_samples_leaf = min_samples_leaf,
            max_features = max_features,
            max_samples = max_samples,
            bootstrap = bootstrap,
            random_state = 42)

    rf_temp.fit(X_train, y_train_temp)

    train_preds_temp = rf_temp.predict(X_train)
    train_rmse = np.sqrt(mean_squared_error(y_train_temp, train_preds_temp))

    val_preds_temp = rf_temp.predict(X_val)
    val_rmse = np.sqrt(mean_squared_error(y_temp_val, val_preds_temp))
    print(f"\thyperparameters: {n_estimators}, {regressor_criterion}, {max_depth}, {min_samples_split}, {min_samples_leaf}, {max_features}, {max_samples}, {bootstrap}")
    print(f"\tTrain RMSE: {train_rmse:.4f}, Val RMSE: {val_rmse:.4f}")

    return rf_temp, train_rmse, val_rmse

def train_weather_rf_classifier_model(
        X_train, y_train_rain, 
        X_val, y_rain_val, 
        n_estimators, classifier_criterion, 
        max_depth, min_samples_split, min_samples_leaf, 
        max_features, max_samples, bootstrap, rf_flag='scikit'):

    if rf_flag == 'cuml':
        rf_rain = cuRFC(
            n_estimators = n_estimators,
            split_criterion = classifier_criterion,
            max_depth = max_depth,
            min_samples_split = min_samples_split,
            min_samples_leaf = min_samples_leaf,
            max_features = max_features,
            max_samples = max_samples,
            bootstrap = bootstrap,
            random_state = 42)
    else:
        rf_rain = RandomForestClassifier(
            n_estimators = n_estimators,
            criterion=classifier_criterion,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            max_samples=max_samples,
            bootstrap=bootstrap,
            random_state = 42)

    rf_rain.fit(X_train, y_train_rain)

    train_preds_rain = rf_rain.predict(X_train)
    train_acc = accuracy_score(y_train_rain, train_preds_rain)

    val_preds_rain = rf_rain.predict(X_val)
    val_acc  = accuracy_score(y_rain_val, val_preds_rain)

    print(f"\thyperparameters: {n_estimators}, {classifier_criterion}, {max_depth}, {min_samples_split}, {min_samples_leaf}, {max_features}, {max_samples}, {bootstrap}")
    print(f"\tTrain Accuracy: {train_acc:.4f}, Val Accuracy: {val_acc:.4f}")

    return rf_rain, train_acc, val_acc

def evaluate_weather_rf_model(rf_temp, rf_rain, X_test, y_temp_labels, y_rain_labels):
    y_temp_preds = rf_temp.predict(X_test)
    y_rain_preds = rf_rain.predict(X_test)

    regression_results(y_temp_labels, y_temp_preds)
    regression_plot(y_temp_labels, y_temp_preds)

    classification_results(y_rain_labels, y_rain_preds)
    classification_plot(y_rain_labels, y_rain_preds)

    return y_temp_preds, y_rain_preds

In [23]:
def weather_rf_model_gridsearch(rf_flag, features, label_1, label_2, regressor_param_grid, classifier_param_grid):
    
    regressor_iterator_counter = 0
    classifier_iterator_counter = 0
    regressor_num_runs = np.prod([len(v) for v in regressor_param_grid.values()])
    classifier_num_runs = np.prod([len(v) for v in classifier_param_grid.values()])

    reg_df = load_or_create_results_df(
        'rf_regressor_model_results', 
        regressor_param_grid.keys(), 
        ['train_rmse', 'val_rmse']
    )
    class_df = load_or_create_results_df(
        'rf_classifier_model_results', 
        classifier_param_grid.keys(), 
        ['train_acc', 'val_acc']
    )

    X_train, y_temp_train, y_rain_train, X_val, y_temp_val, y_rain_val = weather_rf_train_val_dataset(features, label_1, label_2)

    for params in itertools.product(*regressor_param_grid.values()):
        param_dict = dict(zip(regressor_param_grid.keys(), params))

        regressor_iterator_counter += 1

        if check_if_run_exists(reg_df, param_dict):
            print(f"Run {regressor_iterator_counter}/{regressor_num_runs}: skipping existing rf regressor params.")
            continue

        train_rmse, val_rmse = train_weather_rf_regressor_model(
            X_train, y_temp_train,
            X_val, y_temp_val,
            n_estimators=param_dict['n_estimators'],
            regressor_criterion=param_dict['regressor_criterion'],
            max_depth=param_dict['max_depth'],
            min_samples_split=param_dict['min_samples_split'],
            min_samples_leaf=param_dict['min_samples_leaf'],
            max_features=param_dict['max_features'],
            max_samples=param_dict['max_samples'],
            bootstrap=param_dict['bootstrap'],
            rf_flag=rf_flag
        )

        param_dict.update({
            'train_rmse': train_rmse,
            'val_rmse': val_rmse
        })
        
        append_result_to_csv('rf_regressor_model_results', param_dict)

        print(f'Run {regressor_iterator_counter}/{regressor_num_runs} completed for regressor')

    for params in itertools.product(*classifier_param_grid.values()):
        param_dict = dict(zip(classifier_param_grid.keys(), params))

        classifier_iterator_counter += 1

        if check_if_run_exists(class_df, param_dict):
            print(f"Run {classifier_iterator_counter}/{classifier_num_runs}: skipping existing rf classifier params.")
            continue 

        train_acc, val_acc = train_weather_rf_classifier_model(
            X_train, y_rain_train,
            X_val, y_rain_val,
            n_estimators=param_dict['n_estimators'],
            classifier_criterion=param_dict['classifier_criterion'],
            max_depth=param_dict['max_depth'],
            min_samples_split=param_dict['min_samples_split'],
            min_samples_leaf=param_dict['min_samples_leaf'],
            max_features=param_dict['max_features'],
            max_samples=param_dict['max_samples'],
            bootstrap=param_dict['bootstrap'],
            rf_flag=rf_flag
        )

        param_dict.update({
            'train_acc': train_acc,
            'val_acc': val_acc
        })
        
        append_result_to_csv('rf_classifier_model_results', param_dict)
        
        print(f'Run {classifier_iterator_counter}/{classifier_num_runs} completed for classifier')

def weather_rf_model_reevaluate(rf_flag, features, label_1, label_2):

    X_train, y_temp_train, y_rain_train, X_val, y_temp_val, y_rain_val = weather_rf_train_val_dataset(features, label_1, label_2)
    X_test, y_temp_test, y_rain_test = weather_rf_test_dataset(features, label_1, label_2)

    best_regressor_run = get_best_hyperparameters_from_csv(
        'rf_regressor_model_results',
        metric_column='val_rmse',
        mode='min'
    )

    best_classifier_run = get_best_hyperparameters_from_csv(
        'rf_classifier_model_results',
        metric_column='val_acc',
        mode='max'
    )

    if best_regressor_run:
        regressor_params = {k: v for k, v in best_regressor_run.items() if k not in ['train_rmse', 'val_rmse']}
        if 'bootstrap' in regressor_params:
             regressor_params['bootstrap'] = bool(regressor_params['bootstrap'])
        best_model_temp, _, _ = train_weather_rf_regressor_model(X_train, y_temp_train, X_val, y_temp_val, **regressor_params, rf_flag=rf_flag)

    if best_classifier_run:
        classifier_params = {k: v for k, v in best_classifier_run.items() if k not in ['train_acc', 'val_acc']}
        if 'bootstrap' in classifier_params:
             classifier_params['bootstrap'] = bool(classifier_params['bootstrap'])
        best_model_rain, _, _ = train_weather_rf_classifier_model(X_train, y_rain_train, X_val, y_rain_val, **classifier_params, rf_flag=rf_flag)

    y_temp_preds, y_rain_preds = evaluate_weather_rf_model(best_model_temp, best_model_rain, X_test, y_temp_test, y_rain_test)
    
    rf_model_dict = {
        'rf_temp_model' : best_model_temp,
        'rf_rain_model' : best_model_rain,
        'rf_y_temp_labels' : y_temp_test,
        'rf_y_temp_preds' : y_temp_preds,
        'rf_y_rain_labels' : y_rain_test,
        'rf_y_rain_preds' : y_rain_preds,
        'rf_temp_hyperparameters' : regressor_params,
        'rf_rain_hyperparameters' : classifier_params
    }

    save_with_pickle(rf_model_dict, 'rf_model_results.pkl')

    return rf_model_dict

## Ensemble method

In [26]:
def weather_ensemble_test_dataset(features, label_1, label_2, batch_size=128, num_workers=0):
    rf_X_test = test_df[features].values
    rf_y_temp_test = test_df[label_1].values
    rf_y_rain_test = test_df[label_2].values

    lstm_X_test = torch.tensor(test_df[features].values, dtype=torch.float32)
    lstm_y_temp_test = torch.tensor(test_df[label_1].values, dtype=torch.float32).unsqueeze(1)
    lstm_y_rain_test = torch.tensor(test_df[label_2].values, dtype=torch.float32).unsqueeze(1)

    lstm_test_loader = DataLoader(WeatherLSTMDataset(lstm_X_test, lstm_y_temp_test, lstm_y_rain_test), batch_size=batch_size, num_workers=num_workers)

    return rf_X_test, rf_y_temp_test, rf_y_rain_test, lstm_test_loader

def evaluate_weather_ensemble_model(rf_temp, rf_rain, lstm_model, X_test, test_loader, rf_to_lstm_temp_weight, rf_to_lstm_rain_weight):
    """
    rf_temp: trained RandomForestRegressor
    rf_rain: trained RandomForestClassifier
    lstm_model: trained LSTM model
    """

    rf_y_temp_preds = rf_temp.predict(X_test)
    rf_y_rain_preds = rf_rain.predict(X_test)
    
    lstm_model.eval()
    lstm_y_temp_preds = []
    lstm_y_rain_preds = []
    lstm_y_temp_labels = []
    lstm_y_rain_labels = []

    with torch.no_grad():
        for xb, (y_temp, y_rain) in test_loader:
            xb, y_temp, y_rain = xb.to(device), y_temp.to(device), y_rain.to(device)

            temp_output, rain_output = lstm_model(xb)

            predicted_labels = (torch.sigmoid(rain_output) >= 0.5).float()
            test_correct += (predicted_labels == y_rain).sum().item()
            test_total += y_rain.size(0)
            
            lstm_y_rain_preds.extend(predicted_labels.cpu().numpy())
            lstm_y_rain_labels.extend(y_rain.cpu().numpy())

            lstm_y_temp_preds.extend(temp_output.cpu().numpy())
            lstm_y_temp_labels.extend(y_temp.cpu().numpy())
            
    ensemble_y_temp_preds = rf_to_lstm_temp_weight * rf_y_temp_preds + (1 - rf_to_lstm_temp_weight) * lstm_y_temp_preds

    preds_rain_prob = rf_to_lstm_rain_weight * rf_y_rain_preds + (1 - rf_to_lstm_rain_weight) * lstm_y_rain_preds
    ensemble_y_rain_preds = (preds_rain_prob > 0.5).astype(int)
    
    return ensemble_y_temp_preds, ensemble_y_rain_preds

def weather_ensemble_model_gridsearch(rf_temp, rf_rain, lstm_model, features, label_1, label_2, param_grid=None):
    ensemble_iteration_counter = 0

    X_test, y_temp_labels, y_rain_labels, test_loader = weather_ensemble_test_dataset(
        features, label_1, label_2,
        batch_size=128,
        num_workers=0
    )

    ensemble_df = load_or_create_results_df(
        'ensemble_model_result', 
        param_grid.keys(), 
        ['test_rmse', 'test_accuracy']
    )

    num_runs = np.prod([len(v) for v in param_grid.values()])
    for params in itertools.product(*param_grid.values()):
        ensemble_iteration_counter += 1
        param_dict = dict(zip(param_grid.keys(), params))

        if check_if_run_exists(ensemble_df, param_dict):
            print(f"Run {ensemble_iteration_counter}/{num_runs}: skipping existing lstm params.")
            continue

        ensemble_y_temp_preds, ensemble_y_rain_preds = evaluate_weather_ensemble_model(
            rf_temp, rf_rain, lstm_model, X_test, test_loader, 
            param_dict['rf_to_lstm_rain_weight'],
            param_dict['rf_to_lstm_temp_weight']
        )

        rmse = np.sqrt(mean_squared_error(y_temp_labels, ensemble_y_temp_preds))
        accuracy = accuracy_score(y_rain_labels, ensemble_y_rain_preds)
        
        param_dict.update({
            'test_rmse': rmse,
            'test_accuracy': accuracy
        })

        append_result_to_csv('ensemble_model_result', param_dict)


def weather_ensemble_model_reevaluate(rf_temp, rf_rain, lstm_model, features, label_1, label_2):

    X_test, y_temp_labels, y_rain_labels, test_loader = weather_ensemble_test_dataset(
        features, label_1, label_2,
        batch_size=128,
        num_workers=0
    )

    best_ensemble_run = get_best_lstm_hyperparameters(
        'lstm_model_results',
        weight=0.5
    )

    ensemble_y_temp_preds, ensemble_y_rain_preds = evaluate_weather_ensemble_model(
        rf_temp, rf_rain, lstm_model, X_test, test_loader, 
        best_ensemble_run['rf_to_lstm_rain_weight'],
        best_ensemble_run['rf_to_lstm_temp_weight']
    )

    regression_results(y_temp_labels, ensemble_y_temp_preds)
    regression_plot(y_temp_labels, ensemble_y_temp_preds)

    classification_results(y_rain_labels, ensemble_y_rain_preds)
    classification_plot(y_rain_labels, ensemble_y_rain_preds)

    ensemble_model_dict = {
        'ensemble_y_temp_model' : rf_temp,
        'ensemble_y_rain_model' : rf_rain,
        'ensemble_lstm_model' : lstm_model,
        'ensemble_y_temp_labels' : y_temp_labels,
        'ensemble_y_temp_preds' : ensemble_y_temp_preds,
        'ensemble_y_rain_labels' : y_rain_labels,
        'ensemble_y_rain_preds' : ensemble_y_rain_preds,
        'ensemble_hyperparameters' : best_ensemble_run
    }

    save_with_pickle(ensemble_model_dict, 'ensemble_model_results.pkl')

    return ensemble_model_dict

## Execution cell

In [27]:
def search_best_models(models_to_search):
    features = [
            'pressure','dew_point','humidity','wind_speed',
            'gust','wind_chill','uv_index','feels_like_temperature',
            'visibility','solar_radiation', 'precipitation_total',
            'precipitation_total_1h_avg', 'precipitation_total_1d_avg',
            'precipitation_rate_1h_avg', 'precipitation_rate_1d_avg',
            'temperature_1h_avg', 'temperature_1d_avg', 'precipitation_rate'
        ]

    for model_name in models_to_search:
        if model_name == 'baseline':
            temperature_features = features
            temperature_labels = 'temperature'

            temperature_linear_param_grid = {
                'epochs'         : [50],
                'batch_size'     : [128],
                'lr'             : [0.001],
            }

            # temperature_linear_param_grid = {
            #     'epochs'         : [10, 25, 50],
            #     'batch_size'     : [128, 256, 1024],
            #     'lr'             : [0.01, 0.005, 0.001],
            # }

            best_temp_model, y_labels, y_preds = temperature_linear_model_gridsearch(temperature_features, temperature_labels, param_grid=temperature_linear_param_grid)

            base_temp_data = {
                'base_temp_model' : best_temp_model,
                'base_y_temp_labels' : y_labels,
                'base_y_temp_preds' : y_preds,
            }

            save_with_pickle(base_temp_data, 'base_temp_model_results.pkl')

            rainfall_features = [
                'pressure','dew_point','humidity','wind_speed',
                'gust','wind_chill','uv_index','feels_like_temperature',
                'visibility','solar_radiation', 'precipitation_total',
                'precipitation_total_1h_avg', 'precipitation_total_1d_avg',
                'precipitation_rate_1h_avg', 'precipitation_rate_1d_avg',
                'temperature_1h_avg', 'temperature_1d_avg', 'temperature'
            ]
            rainfall_labels = 'rain_label'

            rainfall_binary_param_grid = {
                'epochs'         : [50],
                'batch_size'     : [256],
                'lr'             : [0.01],
                'class_weight'   : [False]
            }

            # rainfall_binary_param_grid = {
            #     'epochs'         : [10, 25, 50],
            #     'batch_size'     : [128, 256, 1024],
            #     'lr'             : [0.01, 0.005, 0.001],
            #     'class_weight'   : [True, False]
            # }

            best_rain_model, y_labels, y_preds = rainfall_binary_model_gridsearch(rainfall_features, rainfall_labels, param_grid=rainfall_binary_param_grid)

            base_rain_data = {
                'base_rain_model' : best_rain_model,
                'base_y_rain_labels' : y_labels,
                'base_y_rain_preds' : y_preds,
            }

            save_with_pickle(base_rain_data, 'base_rain_model_results.pkl')

        elif model_name == 'lstm':
            weather_lstm_param_grid = {
                'epochs'         : [10, 25, 50],
                'batch_size'     : [512, 1024],
                'lr'             : [0.01, 0.001],
                'sequence_length': [7, 14],
                'hidden_size'    : [64, 128],
                'num_layers'     : [1, 2],
                'dropout'        : [0.0, 0.1],
                'class_weight'   : [True, False]
            }

            weather_lstm_model_gridsearch(
                features = features, 
                label_1 = 'temperature',
                label_2 = 'rain_label',
                param_grid=weather_lstm_param_grid
            )

            lstm_model_dict = weather_lstm_model_reevaluate(
                features = features, 
                label_1 = 'temperature',
                label_2 = 'rain_label'
            )

        elif model_name == 'rf':
            # scikit lib hyperparameters
            # https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html
            sk_weather_rf_regressor_param_grid = {
                'n_estimators'          : [1000, 10000, 20000, 100000],
                'regressor_criterion'   : ["squared_error", "absolute_error", "friedman_mse", "poisson"],
                'max_depth'             : [None],
                'min_samples_split'     : [2, 5, 10],
                'min_samples_leaf'      : [1, 2, 4],
                'max_features'          : ['sqrt', 'log2'],
                'max_samples'           : [None, 128, 512, 1024],
                'bootstrap'             : [True] # Bagging to avoid data imbalance
            }

            # https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html
            sk_weather_rf_classifier_param_grid = {
                'n_estimators'          : [1000, 10000, 20000, 100000],
                'classifier_criterion'  : ['gini', 'entropy', 'log_loss'],
                'max_depth'             : [None],
                'min_samples_split'     : [2, 5, 10],
                'min_samples_leaf'      : [1, 2, 4],
                'max_features'          : ['sqrt', 'log2'],
                'max_samples'           : [None, 128, 512, 1024],
                'bootstrap'             : [True] # Bagging to avoid data imbalance
            }

            # rapid hyperparameter tuning lib hyperparameters
            # https://docs.rapids.ai/api/cuml/stable/api/#cuml.ensemble.RandomForestRegressor

            # memory leak issue on the upstream library, https://github.com/rapidsai/cuml/issues/7089
            # reduce n_estimators for workaround
            cuml_weather_rf_regressor_param_grid = {
                'n_estimators'          : [100, 500, 1000],
                'regressor_criterion'   : ["mse", "gamma", "inverse_gaussian", "poisson"],
                'max_depth'             : [1000, 10000, 100000],
                'min_samples_split'     : [2, 5, 10],
                'min_samples_leaf'      : [1, 2, 4],
                'max_features'          : ['sqrt', 'log2', None],
                'max_samples'           : [1.0],
                'bootstrap'             : [True] # Bagging to avoid data imbalance
            }

            # https://docs.rapids.ai/api/cuml/stable/api/#cuml.ensemble.RandomForestClassifier
            cuml_weather_rf_classifier_param_grid = {
                'n_estimators'          : [100, 500, 1000],
                'classifier_criterion'  : ['gini', 'entropy'],
                'max_depth'             : [1000, 10000, 100000],
                'min_samples_split'     : [2, 5, 10],
                'min_samples_leaf'      : [1, 2, 4],
                'max_features'          : ['sqrt', 'log2', None],
                'max_samples'           : [1.0],
                'bootstrap'             : [True] # Bagging to avoid data imbalance
            }

            # switch between scikit-learn or cuml
            # make sure to have the respective libraries and cuda version +installed
            rf_flag = 'cuml'  # 'scikit' or 'cuml'

            if rf_flag == 'cuml':
                weather_rf_regressor_param_grid = cuml_weather_rf_regressor_param_grid
                weather_rf_classifier_param_grid = cuml_weather_rf_classifier_param_grid
            else:
                weather_rf_regressor_param_grid = sk_weather_rf_regressor_param_grid
                weather_rf_classifier_param_grid = sk_weather_rf_classifier_param_grid

            weather_rf_model_gridsearch(
                rf_flag = rf_flag,
                features = features, 
                label_1 = 'temperature',
                label_2 = 'rain_label',
                regressor_param_grid = weather_rf_regressor_param_grid,
                classifier_param_grid = weather_rf_classifier_param_grid)

            rf_model_dict = weather_rf_model_reevaluate(
                rf_flag = rf_flag,
                features = features, 
                label_1 = 'temperature',
                label_2 = 'rain_label')

        elif model_name == 'hybrid':
            try:
                rf_model_dict = load_with_pickle('rf_model_results.pkl')
                lstm_model_dict = load_with_pickle('lstm_model_results.pkl')
                ensemble_param_grid = {
                    'rf_to_lstm_rain_weight' : [0.2, 0.4, 0.6, 0.8],
                    'rf_to_lstm_temp_weight' : [0.2, 0.4, 0.6, 0.8],
                }
                weather_ensemble_model_gridsearch(
                    rf_temp = rf_model_dict['rf_temp_model'], 
                    rf_rain = rf_model_dict['rf_rain_model'], 
                    lstm_model = lstm_model_dict['lstm_model'],
                    features = features, 
                    label_1 = 'temperature',
                    label_2 = 'rain_label',
                    param_grid=ensemble_param_grid
                )
                weather_ensemble_model_reevaluate(
                    rf_temp = rf_model_dict['rf_temp_model'], 
                    rf_rain = rf_model_dict['rf_rain_model'], 
                    lstm_model = lstm_model_dict['lstm_model'],
                    features = features, 
                    label_1 = 'temperature',
                    label_2 = 'rain_label'
                )
            except FileNotFoundError as e:
                print(e)
                print("Model file not found, please ensure you have run the 'rf' and 'lstm' model searches first.\n")
        else:
            print(f"Model {model_name} not recognized.")
            continue

In [ ]:
# Main run function
# search_best_models(['baseline', 'lstm', 'rf', 'hybrid'])
search_best_models(['lstm', 'rf', 'hybrid'])

RuntimeError: input.size(-1) must be equal to input_size. Expected 17, got 18